## Cell 1 — Imports & Config

In [1]:
import json, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.linalg import expm, solve_discrete_are, inv
from scipy.optimize import minimize as sp_minimize
from tqdm.notebook import tqdm

np.set_printoptions(precision=6, suppress=True, linewidth=120)

HOME       = "./"
MODEL_JSON = "digester_prof_final_v2.json"
DT         = 5.0
nx         = 4
PUMP_OFF   = pd.Timestamp("2026-02-15 20:13:00", tz="UTC")
CHUNK_H    = 12
TIME_LIMIT = 25 * 60

# Training window: Feb 22 -> Mar 4 (clean stable regime)
TRAIN_START = pd.Timestamp("2026-02-22 00:00:00", tz="UTC")
TRAIN_END   = pd.Timestamp("2026-03-04 00:00:00", tz="UTC")
VAL_END     = pd.Timestamp("2026-03-06 00:00:00", tz="UTC")  # 48h validation

BG='#0b1120'; PNL='#111827'; GRD='#1e293b'; TXT='#94a3b8'; TTL='#e2e8f0'; SIM='#22d3ee'
_fc = [0]

def show_fig(fig, name="fig"):
    fn = f"v3_{name}.png"
    fig.savefig(fn, dpi=120, facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {fn}")

def r2(a, p):
    ss = np.sum((a-p)**2); st = np.sum((a-a.mean())**2)
    return 1 - ss/st if st > 0 else float('nan')

def rmse(a, p): return np.sqrt(np.mean((a-p)**2))

def sty(ax, yl):
    ax.set_facecolor(PNL); ax.set_ylabel(yl, color=TXT, fontsize=10)
    ax.tick_params(colors=TXT, labelsize=9)
    for s in ax.spines.values(): s.set_color(GRD)
    ax.grid(True, alpha=0.2, color=GRD, linewidth=0.5)

def discretize(A, B, G, dt):
    A_d = expm(A * dt)
    M = np.zeros((nx+3, nx+3)); M[:nx,:nx]=A*dt; M[:nx,nx:]=B*dt
    B_d = expm(M)[:nx, nx:]
    M2 = np.zeros((nx+1, nx+1)); M2[:nx,:nx]=A*dt; M2[:nx,nx]=G*dt
    G_d = expm(M2)[:nx, nx]
    return A_d, B_d, G_d

def design_observer_dare(A_d, H, Q_obs, R_obs):
    P = solve_discrete_are(A_d.T, H.T, Q_obs, R_obs)
    S = H @ P @ H.T + R_obs
    L = P @ H.T @ inv(S)
    return L, np.linalg.eigvals(A_d - L @ H), P

print("  Imports OK")
print(f"  Train: {TRAIN_START.date()} -> {TRAIN_END.date()}")
print(f"  Valid: {TRAIN_END.date()} -> {VAL_END.date()}")


  Imports OK
  Train: 2026-02-22 -> 2026-03-04
  Valid: 2026-03-04 -> 2026-03-06


## Cell 2 — Load Model & Data

In [2]:
# load fixed B from v2
with open(MODEL_JSON) as f: M_old = json.load(f)
B_fixed = np.array(M_old['B'])
sn = ["EC","pH","h1","h2"]
print("  B (fixed from v2):")
for i in range(4):
    print(f"    {sn[i]:>4s} " + "".join(f"{B_fixed[i,j]:>14.6f}" for j in range(3)))

def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").drop_duplicates(subset="time").set_index("time")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

def load_all():
    raw = {
        "EC_mS":           load_csv(HOME+"Digester Tank 1-cloud_EC_mS.csv"),
        "pH":              load_csv(HOME+"Digester Tank 1-cloud_pH.csv"),
        "Temp_C":          load_csv(HOME+"Digester Tank 1-cloud_Temp_C.csv"),
        "ORP_mV":          load_csv(HOME+"Digester Tank 1-cloud_ORP_mV.csv"),
        "MFC_mV":          load_csv(HOME+"Digester Tank 1-cloud_MFC_mV.csv"),
        "Q_urine_mL":      load_csv(HOME+"Digester Tank 1-cloud_Q_urine_mL.csv"),
        "Q_acetate_mL":    load_csv(HOME+"Digester Tank 1-cloud_Q_spir_mL.csv"),
        "Q_phosphoric_mL": load_csv(HOME+"Digester Tank 1-cloud_Q_kombucha_mL.csv"),
    }
    for nm, d in raw.items():
        print(f"    {nm:>20s}: {len(d):>6d} rows  [{d.index.min()} -> {d.index.max()}]")
    return raw

def resample_merge(dfs, freq="5s"):
    all_starts  = [df.index.min() for df in dfs.values()]
    sensor_ends = [df.index.max() for n, df in dfs.items() if "Q_" not in n]
    idx = pd.date_range(start=max(all_starts).floor(freq),
                        end=min(sensor_ends).ceil(freq), freq=freq, tz="UTC")
    res = {}; orp_real = None; mfc_real = None
    for name, df in dfs.items():
        if "Q_" in name:
            s = df["value"].resample(freq).sum().reindex(idx).fillna(0.0)
        elif name in ("ORP_mV", "MFC_mV"):
            s = df["value"].resample(freq).mean().reindex(idx)
            real = s.notna()
            if name == "ORP_mV": orp_real = s.index[real]
            else:                mfc_real = s.index[real]
            s = s.ffill()
        else:
            s = df["value"].resample(freq).mean().reindex(idx).interpolate(method="time")
        res[name] = s
    merged = pd.DataFrame(res, index=idx).dropna()
    off = merged.index >= PUMP_OFF
    if off.any():
        merged.loc[off, "Q_urine_mL"]      = 0.0
        merged.loc[off, "Q_acetate_mL"]    = 0.0
        merged.loc[off, "Q_phosphoric_mL"] = 0.0
        print(f"  PUMP SHUTOFF: zeroed {off.sum()} rows after {PUMP_OFF}")
    merged["_ORP_real"] = merged.index.isin(orp_real) if orp_real is not None else True
    merged["_MFC_real"] = merged.index.isin(mfc_real) if mfc_real is not None else True
    return merged

print("\n  Loading data ...")
raw = load_all()
df_full = resample_merge(raw, freq=f"{int(DT)}s")

# slice to training and validation windows
df_train = df_full[(df_full.index >= TRAIN_START) & (df_full.index < TRAIN_END)]
df_val   = df_full[(df_full.index >= TRAIN_END)   & (df_full.index < VAL_END)]

ec0   = float(df_train["EC_mS"].iloc[:720].mean())
ph0   = float(df_train["pH"].iloc[:720].mean())
temp0 = float(df_train["Temp_C"].iloc[:720].mean())

print(f"\n  Train: {df_train.index.min()} -> {df_train.index.max()}")
print(f"  Valid: {df_val.index.min()} -> {df_val.index.max()}")
print(f"  Train pts: {len(df_train)}  |  Val pts: {len(df_val)}")
print(f"  Nominal: EC0={ec0:.4f}  pH0={ph0:.4f}  T0={temp0:.2f}")


  B (fixed from v2):
      EC       0.003243     -0.008060      0.000579
      pH       0.000206      0.000563     -0.000205
      h1       0.000007      0.000012      0.000016
      h2       0.000043      0.000081     -0.000008

  Loading data ...
                   EC_mS: 184108 rows  [2026-02-22 00:00:00.279035822+00:00 -> 2026-03-08 20:54:24.146958022+00:00]
                      pH: 183649 rows  [2026-02-22 00:00:00.279035822+00:00 -> 2026-03-08 20:54:29.274455150+00:00]
                  Temp_C:  43151 rows  [2026-02-22 00:00:25.363279944+00:00 -> 2026-03-08 20:54:14.122797789+00:00]
                  ORP_mV: 137852 rows  [2026-02-22 00:00:00.279035822+00:00 -> 2026-03-08 20:54:24.146958022+00:00]
                  MFC_mV: 126540 rows  [2026-02-22 00:00:05.430483283+00:00 -> 2026-03-08 20:54:24.146958022+00:00]
              Q_urine_mL:   8372 rows  [2026-02-22 00:00:55.412131031+00:00 -> 2026-03-08 20:49:59.184573337+00:00]
            Q_acetate_mL:   8059 rows  [2026-02-22 00:0

## Cell 3 — Build Training Chunks (12h)

In [3]:
chunk_size = int(CHUNK_H * 3600 / DT)
chunks_df  = []
for s in range(0, len(df_train)-chunk_size+1, chunk_size):
    chunks_df.append(df_train.iloc[s:s+chunk_size])
rem = len(df_train) - len(chunks_df)*chunk_size
if rem > chunk_size//2:
    chunks_df.append(df_train.iloc[len(chunks_df)*chunk_size:])

CA = []
for ch in chunks_df:
    CA.append({
        'EC':  ch['EC_mS'].values.astype(np.float64),
        'pH':  ch['pH'].values.astype(np.float64),
        'U':   ch[['Q_urine_mL','Q_acetate_mL','Q_phosphoric_mL']].values.astype(np.float64),
        'T':   ch['Temp_C'].values.astype(np.float64),
        'ORP': ch['ORP_mV'].values.astype(np.float64),
        'MFC': ch['MFC_mV'].values.astype(np.float64),
        'N':   len(ch),
    })
print(f"  {len(CA)} chunks x {CHUNK_H}h ({chunk_size} pts each)")


  20 chunks x 12h (8640 pts each)


## Cell 4 — Optimise A & G (B fixed, 25 min)
Run this cell multiple times if needed — it warm starts each time.

In [10]:
# ── pre-extract training arrays ONCE outside cost function ───────────
_EC  = df_train["EC_mS"].values.copy()
_pH  = df_train["pH"].values.copy()
_U   = df_train[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values.copy()
_T   = df_train["Temp_C"].values.copy()
_N   = len(_EC)

# pre-allocate simulation buffer ONCE — reused every eval
_x_buf = np.zeros((_N, nx), dtype=np.float64)

print(f"  Training arrays pre-extracted: {_N} pts")
print(f"  Simulation buffer: {_x_buf.nbytes / 1e6:.1f} MB allocated once")

def sim_continuous_fast(A, G, h_init, Tr):
    """Reuses pre-allocated buffer — no new memory each call."""
    x0 = np.array([ec0, ph0, h_init[0], h_init[1]])
    _x_buf[0,0] = _EC[0]; _x_buf[0,1] = _pH[0]
    _x_buf[0,2] = h_init[0]; _x_buf[0,3] = h_init[1]
    for k in range(_N-1):
        _x_buf[k+1] = _x_buf[k] + DT*(A@(_x_buf[k]-x0) + B_fixed@_U[k] + G*(_T[k]-Tr))
    ec_err = np.sum((_x_buf[:,0] - _EC)**2)
    ph_err = np.sum((_x_buf[:,1] - _pH)**2)
    return ec_err + ph_err

_ne=[0]; _best=[1e30]; _t=[0.]; _bat=[0]; _bx=[None]
class _Stop(Exception): pass

def cost(theta):
    _ne[0] += 1
    if time.time()-_t[0] > TIME_LIMIT: raise _Stop()
    A_  = theta[:16].reshape(4,4)
    G_  = theta[16:20]
    h0_ = theta[20:22]
    Tr_ = theta[22]
    # cheap stability check before running sim
    evs = np.linalg.eigvals(A_)
    max_ev = max(ev.real for ev in evs)
    if max_ev > 0.01:
        tot = _best[0] + 1e6 * max_ev**2
        return tot
    tot = sim_continuous_fast(A_, G_, h0_, Tr_)
    # soft stability penalty
    for ev in evs:
        if ev.real > 0: tot += 1e6 * ev.real**2
    if tot < _best[0]*0.9995: _best[0]=tot; _bat[0]=_ne[0]; _bx[0]=theta.copy()
    elif tot < _best[0]:      _best[0]=tot; _bx[0]=theta.copy()
    if _ne[0] % 20 == 0:
        print(f"  eval {_ne[0]:>5d}  cost={tot:.2f}  best={_best[0]:.2f}  "
              f"stall={_ne[0]-_bat[0]}  ({time.time()-_t[0]:.0f}s)", flush=True)
    return tot

# warm start from previous run if A exists, else cold start from v2
try:
    theta_warm = np.concatenate([A.ravel(), G, h0, [Tr]])
    print("  Warm start from previous run")
except NameError:
    theta_warm = np.concatenate([np.array(M_old['A']).ravel(),
                                  np.array(M_old['G']),
                                  np.array(M_old['h0']),
                                  [float(M_old['T_ref'])]])
    print("  Cold start from v2")

_t[0]=time.time(); ic=cost(theta_warm)
_ne[0]=0; _best[0]=ic; _bat[0]=0; _bx[0]=theta_warm.copy(); _t[0]=time.time()
print(f"  Initial cost: {ic:.2f}")
print(f"  Optimising continuously for 25 min ...")

try:
    res=sp_minimize(cost, theta_warm, method='Nelder-Mead',
                    options={'maxiter':100000,'xatol':1e-10,'fatol':1e-10,'adaptive':True})
    bx=res.x
    print(f"\n  CONVERGED: cost={res.fun:.4f}  evals={_ne[0]}  ({time.time()-_t[0]:.0f}s)")
except _Stop:
    bx=_bx[0]
    print(f"\n  TIME LIMIT: best={_best[0]:.4f}  evals={_ne[0]}  ({time.time()-_t[0]:.0f}s)")
except KeyboardInterrupt:
    bx=_bx[0]
    print(f"\n  INTERRUPTED: best={_best[0]:.4f}  evals={_ne[0]}")

A=bx[:16].reshape(4,4); G=bx[16:20]; h0=bx[20:22]; Tr=bx[22]; B=B_fixed

# final check using buffer
_x_buf[0,0]=_EC[0]; _x_buf[0,1]=_pH[0]; _x_buf[0,2]=h0[0]; _x_buf[0,3]=h0[1]
x0=np.array([ec0,ph0,h0[0],h0[1]])
for k in range(_N-1):
    _x_buf[k+1] = _x_buf[k] + DT*(A@(_x_buf[k]-x0) + B_fixed@_U[k] + G*(_T[k]-Tr))

print(f"\n  Continuous sim check:")
print(f"    EC  R2={r2(_EC, _x_buf[:,0]):.4f}  RMSE={rmse(_EC, _x_buf[:,0]):.4f}")
print(f"    pH  R2={r2(_pH, _x_buf[:,1]):.4f}  RMSE={rmse(_pH, _x_buf[:,1]):.4f}")
print(f"    EC final={_x_buf[-1,0]:.4f}  pH final={_x_buf[-1,1]:.4f}")

print(f"\n  Eigenvalues of A:")
for i,ev in enumerate(np.linalg.eigvals(A)):
    tc=abs(1/ev.real)/60 if abs(ev.real)>1e-10 else float('inf')
    print(f"    l_{i+1}={ev.real:+.6f} (tau={tc:.1f}min)  stable={ev.real<0}")
print(f"  h0={h0}  T_ref={Tr:.2f}")
print(f"\n  >>> Run cell 4 again for another 25min pass, or proceed to cell 5")

  Training arrays pre-extracted: 172785 pts
  Simulation buffer: 5.5 MB allocated once
  Warm start from previous run
  Initial cost: 185618.70
  Optimising continuously for 25 min ...
  eval    20  cost=188299.60  best=125206.61  stall=13  (64s)
  eval    40  cost=122676.47  best=122676.47  stall=0  (127s)
  eval    60  cost=120392.39  best=118809.61  stall=6  (191s)
  eval    80  cost=120423.56  best=117872.85  stall=11  (254s)
  eval   100  cost=117294.72  best=117294.72  stall=0  (318s)
  eval   120  cost=116676.50  best=116676.50  stall=8  (381s)
  eval   140  cost=117286.06  best=116500.53  stall=6  (444s)
  eval   160  cost=116657.20  best=115813.30  stall=4  (507s)
  eval   180  cost=116289.91  best=115636.19  stall=5  (571s)
  eval   200  cost=115308.60  best=115308.60  stall=13  (634s)
  eval   220  cost=115089.04  best=114899.73  stall=4  (697s)
  eval   240  cost=114933.74  best=114668.38  stall=10  (761s)
  eval   260  cost=114617.78  best=114370.40  stall=10  (824s)
  eva

## Cell 5 — Fit C Matrix

In [11]:
A_d, B_d, G_d = discretize(A, B, G, DT)

axd=[]; aorp=[]; amfc=[]
orp_rm = df_train["_ORP_real"].values
mfc_rm = df_train["_MFC_real"].values

for cd in CA:
    xs = sim_chunk(A, G, h0, cd, Tr)
    xd = xs.copy(); xd[:,0]-=ec0; xd[:,1]-=ph0
    axd.append(xd); aorp.append(cd['ORP']); amfc.append(cd['MFC'])

axd=np.vstack(axd); aorp=np.concatenate(aorp); amfc=np.concatenate(amfc)
om=orp_rm[:len(aorp)]; mm=mfc_rm[:len(amfc)]

co,_,_,_=np.linalg.lstsq(np.column_stack([axd[om],np.ones(om.sum())]),aorp[om],rcond=None)
cm,_,_,_=np.linalg.lstsq(np.column_stack([axd[mm],np.ones(mm.sum())]),amfc[mm],rcond=None)
C_out=np.array([co[:4],cm[:4]]); biases=np.array([co[-1],cm[-1]])

print(f"  C fit on {om.sum()} ORP pts, {mm.sum()} MFC pts")
print(f"\n  NEW C matrix:")
print("      "+"".join(f"{n:>10s}" for n in sn)+"       bias")
print(f"  ORP "+"".join(f"{C_out[0,j]:>10.4f}" for j in range(4))+f"  {biases[0]:>10.4f}")
print(f"  MFC "+"".join(f"{C_out[1,j]:>10.4f}" for j in range(4))+f"  {biases[1]:>10.4f}")

aec=np.concatenate([cd['EC'] for cd in CA]); apc=np.concatenate([cd['pH'] for cd in CA])
aes=np.concatenate([sim_chunk(A,G,h0,cd,Tr)[:,0] for cd in CA])
aps=np.concatenate([sim_chunk(A,G,h0,cd,Tr)[:,1] for cd in CA])
az=(C_out@axd.T).T+biases

print(f"\n  Training metrics:")
for lbl,obs,pred,mask in [("EC",aec,aes,None),("pH",apc,aps,None),
                           ("ORP",aorp,az[:,0],om),("MFC",amfc,az[:,1],mm)]:
    m=mask if mask is not None else np.ones(len(obs),dtype=bool)
    print(f"    {lbl:>4s}  R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}")


  C fit on 108189 ORP pts, 101174 MFC pts

  NEW C matrix:
              EC        pH        h1        h2       bias
  ORP     2.8663 -103.2462  -54.9783  382.1603   -117.1564
  MFC    -4.9211   15.2089  -18.2655  -10.3996    349.4328

  Training metrics:
      EC  R2=0.1758  RMSE=0.7075
      pH  R2=-0.5752  RMSE=0.3263
     ORP  R2=0.0727  RMSE=43.1516
     MFC  R2=0.0080  RMSE=26.7257


## Cell 6 — Redesign Observer (DARE)

In [12]:
H = np.zeros((2,4)); H[0,0]=1; H[1,1]=1
O = np.vstack([H@np.linalg.matrix_power(A_d,i) for i in range(4)])
print(f"  Observability rank: {np.linalg.matrix_rank(O)}/4")

Q_obs=np.diag([0.1,0.1,0.001,0.001]); R_obs=np.diag([0.04,0.01])
try:
    L,obs_eigs,_=design_observer_dare(A_d,H,Q_obs,R_obs); print("  DARE solved")
except Exception as e:
    from scipy.signal import place_poles as pp
    res2=pp(A_d.T,H.T,[0.95,0.93,0.90,0.88]); L=res2.gain_matrix.T
    obs_eigs=np.linalg.eigvals(A_d-L@H); print(f"  Pole placement fallback: {e}")

print(f"\n  L:"); 
for i in range(4): print(f"    {sn[i]:>4s}  {L[i,0]:>12.6f}  {L[i,1]:>12.6f}")
print(f"\n  Observer poles:")
for i,ev in enumerate(obs_eigs):
    mag=abs(ev); tau=-DT/np.log(mag) if 0<mag<1 else float('inf')
    print(f"    p_{i+1}={ev.real:+.6f} |p|={mag:.6f} tau={tau:.1f}s  stable={mag<1}")


  Observability rank: 4/4
  DARE solved

  L:
      EC      0.764140      0.000853
      pH      0.000213      0.916034
      h1      0.000672     -0.000031
      h2      0.000579     -0.000179

  Observer poles:
    p_1=+0.219693 |p|=0.219693 tau=3.3s  stable=True
    p_2=+0.080322 |p|=0.080322 tau=2.0s  stable=True
    p_3=+0.994165 |p|=0.994165 tau=854.4s  stable=True
    p_4=+0.994480 |p|=0.994480 tau=903.3s  stable=True


## Cell 7 — Run Observer on Train & Validation

In [13]:
def run_observer(A_d,B_d,G_d,L,H,h_init,Tr,ec0,ph0,df_data):
    xr=np.array([ec0,ph0,h_init[0],h_init[1]])
    oEC=df_data["EC_mS"].values; opH=df_data["pH"].values
    oTp=df_data["Temp_C"].values
    U=df_data[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values
    N=len(oEC)
    orm=df_data["_ORP_real"].values if "_ORP_real" in df_data.columns else np.ones(N,dtype=bool)
    mfm=df_data["_MFC_real"].values if "_MFC_real" in df_data.columns else np.ones(N,dtype=bool)
    def step(x,temp,u):
        xn=A_d@(x-xr)+xr+B_d@u+G_d*(temp-Tr)
        xn[0]=np.clip(xn[0],0,30); xn[1]=np.clip(xn[1],3,12); return xn
    xp=np.zeros((N,nx)); xc=np.zeros((N,nx)); inn=np.zeros((N,2))
    xc[0]=[oEC[0],opH[0],h_init[0],h_init[1]]; xp[0]=xc[0].copy()
    for k in tqdm(range(1,N), desc="Observer"):
        xpk=step(xc[k-1],oTp[k-1],U[k-1]); xp[k]=xpk
        y=np.array([oEC[k],opH[k]]); innk=y-H@xpk; inn[k]=innk
        xck=xpk+L@innk; xck[0]=np.clip(xck[0],0,30); xck[1]=np.clip(xck[1],3,12); xc[k]=xck
    return xp,xc,inn,orm,mfm

print("  Running observer on training ...")
_,xc_tr,inn_tr,orm_tr,mrm_tr=run_observer(A_d,B_d,G_d,L,H,h0,Tr,ec0,ph0,df_train)
oEC_tr=df_train["EC_mS"].values; opH_tr=df_train["pH"].values
oORP_tr=df_train["ORP_mV"].values; oMFC_tr=df_train["MFC_mV"].values
xd_tr=xc_tr.copy(); xd_tr[:,0]-=ec0; xd_tr[:,1]-=ph0; z_tr=(C_out@xd_tr.T).T+biases

print("\n  TRAINING:")
for lbl,obs,pred,mask in [("EC",oEC_tr,xc_tr[:,0],None),("pH",opH_tr,xc_tr[:,1],None),
                           ("ORP",oORP_tr,z_tr[:,0],orm_tr),("MFC",oMFC_tr,z_tr[:,1],mrm_tr)]:
    m=mask if mask is not None else np.ones(len(obs),dtype=bool)
    print(f"    {lbl:>4s}  R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}")

h_val=xc_tr[-1,2:]
print(f"\n  Handing off h1={h_val[0]:.6f}  h2={h_val[1]:.6f} to validation")
print("  Running observer on validation ...")
_,xc_v,inn_v,orm_v,mrm_v=run_observer(A_d,B_d,G_d,L,H,h_val,Tr,ec0,ph0,df_val)
oEC_v=df_val["EC_mS"].values; opH_v=df_val["pH"].values
oORP_v=df_val["ORP_mV"].values; oMFC_v=df_val["MFC_mV"].values
xd_v=xc_v.copy(); xd_v[:,0]-=ec0; xd_v[:,1]-=ph0; z_v=(C_out@xd_v.T).T+biases

print("\n  VALIDATION:")
for lbl,obs,pred,mask in [("EC",oEC_v,xc_v[:,0],None),("pH",opH_v,xc_v[:,1],None),
                           ("ORP",oORP_v,z_v[:,0],orm_v),("MFC",oMFC_v,z_v[:,1],mrm_v)]:
    m=mask if mask is not None else np.ones(len(obs),dtype=bool)
    print(f"    {lbl:>4s}  R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}")


  Running observer on training ...


Observer:   0%|          | 0/172784 [00:00<?, ?it/s]


  TRAINING:
      EC  R2=0.9911  RMSE=0.0735
      pH  R2=0.9963  RMSE=0.0158
     ORP  R2=-0.6931  RMSE=58.3065
     MFC  R2=0.0318  RMSE=26.4029

  Handing off h1=-0.511775  h2=-0.257514 to validation
  Running observer on validation ...


Observer:   0%|          | 0/34559 [00:00<?, ?it/s]


  VALIDATION:
      EC  R2=1.0000  RMSE=0.0015
      pH  R2=1.0000  RMSE=0.0001
     ORP  R2=nan  RMSE=nan
     MFC  R2=nan  RMSE=nan


/scratch/3660713.1.engineering/ipykernel_2967450/4148607674.py:36: RuntimeWarning: Mean of empty slice.
  ss = np.sum((a-p)**2); st = np.sum((a-a.mean())**2)
/usr3/graduate/millerjd/.local/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr3/graduate/millerjd/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,


## Cell 8 — Plots (saved to PNG)

In [15]:
def plot_4signals(df_data, xc, z, orm, mrm, name, title):
    t=df_data.index
    oEC=df_data["EC_mS"].values; opH=df_data["pH"].values
    oORP=df_data["ORP_mV"].values; oMFC=df_data["MFC_mV"].values
    fig,axes=plt.subplots(4,1,figsize=(16,14),facecolor=BG)
    fig.subplots_adjust(hspace=0.32,left=0.08,right=0.96,top=0.92,bottom=0.05)
    fig.suptitle(title,fontsize=13,color=TTL,fontweight='bold')
    panels=[("EC (mS)",oEC,xc[:,0],None,"#10b981"),("pH",opH,xc[:,1],None,"#f59e0b"),
            ("ORP (mV)",oORP,z[:,0],orm,"#ef4444"),("MFC (mV)",oMFC,z[:,1],mrm,"#a78bfa")]
    for pi,(lbl,obs,pred,mask,clr) in enumerate(panels):
        ax=axes[pi]; sty(ax,lbl)
        ax.plot(t,obs,"-",color=clr,lw=0.8,alpha=0.5,label="Measured")
        ax.plot(t,pred,"-",color=SIM,lw=1.5,alpha=0.9,label="Model")
        m=mask if mask is not None else np.ones(len(obs),dtype=bool)
        ax.text(0.98,0.96,f"R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}",
                transform=ax.transAxes,ha="right",va="top",fontsize=9,color=SIM,fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.3",facecolor=BG,edgecolor=GRD,alpha=0.9))
        leg=ax.legend(fontsize=8,loc="lower right",framealpha=0.8,facecolor=PNL,edgecolor=GRD)
        for lt in leg.get_texts(): lt.set_color(TTL)
    show_fig(fig, name)

plot_4signals(df_train,xc_tr,z_tr,orm_tr,mrm_tr,"training","TRAINING — v3 (Feb 22 - Mar 4)")
plot_4signals(df_val,xc_v,z_v,orm_v,mrm_v,"validation",f"VALIDATION — v3 (Mar 4 - Mar 6)")

# h1/h2 plot
fig,axes=plt.subplots(2,1,figsize=(16,8),facecolor=BG)
fig.subplots_adjust(hspace=0.35,left=0.08,right=0.96,top=0.90,bottom=0.08)
fig.suptitle("Hidden States h1 & h2 — Training Window",fontsize=13,color=TTL,fontweight='bold')
for ri,(si,lbl,clr) in enumerate([(2,"h1","#ef4444"),(3,"h2","#a78bfa")]):
    ax=axes[ri]; sty(ax,lbl); ax.plot(df_train.index,xc_tr[:,si],color=clr,lw=1.0,alpha=0.8)
show_fig(fig, "hidden_states")


  Saved: v3_training.png


/scratch/3660713.1.engineering/ipykernel_2967450/4148607674.py:36: RuntimeWarning: Mean of empty slice.
  ss = np.sum((a-p)**2); st = np.sum((a-a.mean())**2)


  Saved: v3_validation.png
  Saved: v3_hidden_states.png


## Cell 9 — Save digester_prof_final_v3.json

In [16]:
val_m={
    "r2_EC":r2(oEC_v,xc_v[:,0]),"rmse_EC":rmse(oEC_v,xc_v[:,0]),
    "r2_pH":r2(opH_v,xc_v[:,1]),"rmse_pH":rmse(opH_v,xc_v[:,1]),
    "r2_ORP":r2(oORP_v[orm_v],z_v[orm_v,0]),"rmse_ORP":rmse(oORP_v[orm_v],z_v[orm_v,0]),
    "r2_MFC":r2(oMFC_v[mrm_v],z_v[mrm_v,1]),"rmse_MFC":rmse(oMFC_v[mrm_v],z_v[mrm_v,1]),
}
train_m={
    "r2_EC":r2(aec,aes),"rmse_EC":rmse(aec,aes),
    "r2_pH":r2(apc,aps),"rmse_pH":rmse(apc,aps),
    "r2_ORP":r2(aorp[om],az[om,0]),"rmse_ORP":rmse(aorp[om],az[om,0]),
    "r2_MFC":r2(amfc[mm],az[mm,1]),"rmse_MFC":rmse(amfc[mm],az[mm,1]),
}

res_json={
    "version":"v3 — A,G,C retrained Feb22-Mar4; B fixed from v2",
    "dt":DT,"A":A.tolist(),"B":B.tolist(),"G":G.tolist(),
    "A_d":A_d.tolist(),"B_d":B_d.tolist(),"G_d":G_d.tolist(),
    "C":C_out.tolist(),"biases":biases.tolist(),
    "L":L.tolist(),"H":H.tolist(),
    "h0":h0.tolist(),"T_ref":float(Tr),
    "nominal":{"EC_mS":float(ec0),"pH":float(ph0)},
    "train_metrics":train_m,"validation_metrics":val_m,
    "train_window":{"start":str(TRAIN_START),"end":str(TRAIN_END)},
    "val_window":{"start":str(TRAIN_END),"end":str(VAL_END)},
    "pump_off":str(PUMP_OFF),
    "B_source":"fixed from digester_prof_final_v2.json",
}

with open("digester_prof_final_v3.json","w") as f: json.dump(res_json,f,indent=2)
print("  Saved: digester_prof_final_v3.json")
print("\n  VALIDATION METRICS:")
for k,v in val_m.items(): print(f"    {k}: {v:.4f}")


  Saved: digester_prof_final_v3.json

  VALIDATION METRICS:
    r2_EC: 1.0000
    rmse_EC: 0.0015
    r2_pH: 1.0000
    rmse_pH: 0.0001
    r2_ORP: nan
    rmse_ORP: nan
    r2_MFC: nan
    rmse_MFC: nan


/scratch/3660713.1.engineering/ipykernel_2967450/4148607674.py:36: RuntimeWarning: Mean of empty slice.
  ss = np.sum((a-p)**2); st = np.sum((a-a.mean())**2)


In [9]:
# test continuous simulation over full training window
x_cont = np.zeros((len(df_train), nx))
x_cont[0] = [df_train["EC_mS"].iloc[0], df_train["pH"].iloc[0], h0[0], h0[1]]
x0 = np.array([ec0, ph0, h0[0], h0[1]])
U  = df_train[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values
T  = df_train["Temp_C"].values
for k in range(len(df_train)-1):
    x_cont[k+1] = x_cont[k] + DT*(A@(x_cont[k]-x0) + B_fixed@U[k] + G*(T[k]-Tr))

print(f"EC  R2={r2(df_train['EC_mS'].values, x_cont[:,0]):.4f}  RMSE={rmse(df_train['EC_mS'].values, x_cont[:,0]):.4f}")
print(f"pH  R2={r2(df_train['pH'].values,    x_cont[:,1]):.4f}  RMSE={rmse(df_train['pH'].values,    x_cont[:,1]):.4f}")
print(f"EC final: {x_cont[-1,0]:.4f}  pH final: {x_cont[-1,1]:.4f}")

EC  R2=-0.2398  RMSE=0.8677
pH  R2=-3.7531  RMSE=0.5669
EC final: 10.1487  pH final: 8.0902


In [18]:
# CONVERGENCE SIMULATION — initialize observer with wrong h1/h2
# Shows observer converges to true states from bad initial conditions

N_conv = min(len(df_train), 10000)  # first ~14 hours
df_conv = df_train.iloc[:N_conv]

# wrong initial h1/h2 — 100x the true values
h_wrong = np.array([h0[0]*100, h0[1]*100])
h_true  = h0.copy()

print(f"  True h0:  h1={h_true[0]:.6f}  h2={h_true[1]:.6f}")
print(f"  Wrong h0: h1={h_wrong[0]:.6f}  h2={h_wrong[1]:.6f}")

_, xc_wrong, _, _, _ = run_observer(A_d,B_d,G_d,L,H,h_wrong,Tr,ec0,ph0,df_conv)
_, xc_true,  _, _, _ = run_observer(A_d,B_d,G_d,L,H,h_true, Tr,ec0,ph0,df_conv)

settle = int(0.25 * N_conv)  # settled = last 75%
h1_rmse = rmse(xc_true[settle:,2], xc_wrong[settle:,2])
h2_rmse = rmse(xc_true[settle:,3], xc_wrong[settle:,3])

print(f"\n  Settled RMSE (last 75%):")
print(f"    h1 RMSE = {h1_rmse:.6f}")
print(f"    h2 RMSE = {h2_rmse:.6f}")

# plot convergence
fig, axes = plt.subplots(2,1,figsize=(14,8),facecolor=BG)
fig.subplots_adjust(hspace=0.35,left=0.08,right=0.96,top=0.90,bottom=0.08)
fig.suptitle("Observer Convergence — Wrong vs True h1/h2 Initial Conditions (v3)",
             fontsize=13,color=TTL,fontweight='bold')
t_conv = df_conv.index
for ri,(si,lbl,clr) in enumerate([(2,'h1','#ef4444'),(3,'h2','#a78bfa')]):
    ax=axes[ri]; sty(ax,lbl)
    ax.plot(t_conv,xc_true[:,si], color=clr,  lw=1.5, alpha=0.9, label='True init')
    ax.plot(t_conv,xc_wrong[:,si],color='white',lw=1.0,alpha=0.6,ls='--',label='Wrong init')
    leg=ax.legend(fontsize=8,framealpha=0.8,facecolor=PNL,edgecolor=GRD)
    for lt in leg.get_texts(): lt.set_color(TTL)
show_fig(fig,"convergence")

  True h0:  h1=-0.000035  h2=-0.000004
  Wrong h0: h1=-0.003529  h2=-0.000388


Observer:   0%|          | 0/9999 [00:00<?, ?it/s]

Observer:   0%|          | 0/9999 [00:00<?, ?it/s]


  Settled RMSE (last 75%):
    h1 RMSE = 0.003493
    h2 RMSE = 0.000385
  Saved: v3_convergence.png
